# 06 — Evaluation inspection

Pure inspection of an ablation study run. No orchestration here: phases are run via `scripts/run_evaluation_phase.py` and the rollup via `scripts/run_evaluation_rollup.py`. This notebook only **reads** MLflow artifacts and renders comparisons.

Set `ABLATION_ID` in the next cell to the value printed by the phase scripts.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from mlflow.tracking import MlflowClient

from src.tracking import setup_tracking_uri
from src.evaluation.sample import load_eval_sample
from src.evaluation.metrics import check_format_adherence

ABLATION_ID = "REPLACE_ME"  # e.g. "2026-05-21T1430"
EXPERIMENT = "tfm-evaluation"
CONFIG_ORDER = ["A_base", "B_finetuned", "C_base_rag", "D_finetuned_rag"]
FIGURES_DIR = Path("../docs/thesis_latex/figures")

setup_tracking_uri()
client = MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT)
print("experiment_id:", experiment.experiment_id)

In [ ]:
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.ablation_id = '{ABLATION_ID}'",
)
by_kind = {"phase_parent": [], "config_child": [], "rollup": []}
for r in runs:
    by_kind.setdefault(r.data.tags.get("kind", "?"), []).append(r)
children = {r.data.tags["config"]: r for r in by_kind["config_child"]}
rollup = by_kind["rollup"][0] if by_kind["rollup"] else None
print({k: len(v) for k, v in by_kind.items()})
assert set(children) == set(CONFIG_ORDER), f"Missing configs: {set(CONFIG_ORDER) - set(children)}"

## 2. Eval sample inspection

In [ ]:
samples = load_eval_sample(
    indices_path="../data/processed/eval_sample_indices.json",
    dataset_path="../data/processed/dataset",
)
samples_df = pd.DataFrame(samples)
samples_df["trans_len"] = samples_df["transcription"].str.len()
print("n samples:", len(samples_df))
samples_df["medical_specialty"].value_counts()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
samples_df["medical_specialty"].value_counts().plot.barh(ax=axes[0], color="0.4")
axes[0].set_title("Eval sample — specialty distribution")
axes[0].invert_yaxis()
axes[1].hist(samples_df["trans_len"], bins=20, color="0.4", edgecolor="black")
axes[1].set_title("Transcription length (chars)")
plt.tight_layout(); plt.show()

## 3. Per-config record loading

In [ ]:
def load_jsonl_from_run(run_id: str) -> pd.DataFrame:
    local = client.download_artifacts(run_id, "records")
    jsonl_files = list(Path(local).glob("*.jsonl"))
    assert len(jsonl_files) == 1, jsonl_files
    with open(jsonl_files[0]) as f:
        return pd.DataFrame([json.loads(l) for l in f if l.strip()])

per_config = {name: load_jsonl_from_run(run.info.run_id) for name, run in children.items()}
for name, df in per_config.items():
    print(f"{name}: {len(df)} rows | abstained={df['abstained'].sum()} | errors={df['error'].notna().sum()}")

## 4. Side-by-side response comparison

Pick 5–10 sample indices and render a markdown table for qualitative review.

In [ ]:
from IPython.display import Markdown

show_idxs = sorted(per_config["A_base"]["sample_idx"].sample(5, random_state=0).tolist())

def truncate(s, n=400):
    if s is None:
        return "_(abstained / error)_"
    return (s[:n] + "…") if len(s) > n else s

lines = ["| idx | specialty | A | B | C | D | ground_truth |", "|---|---|---|---|---|---|---|"]
for idx in show_idxs:
    row = {n: per_config[n].loc[per_config[n]["sample_idx"] == idx].iloc[0] for n in CONFIG_ORDER}
    spec = row["A_base"]["medical_specialty"]
    gt = row["A_base"]["ground_truth_response"]
    lines.append(
        f"| {idx} | {spec} | {truncate(row['A_base']['response'])} | {truncate(row['B_finetuned']['response'])} | "
        f"{truncate(row['C_base_rag']['response'])} | {truncate(row['D_finetuned_rag']['response'])} | {truncate(gt)} |"
    )
Markdown("\n".join(lines))

## 5. Hallucinated PMIDs — headline finding

In [ ]:
rates = {n: children[n].data.metrics.get("hallucinated_pmids_rate", math.nan) for n in CONFIG_ORDER}
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(CONFIG_ORDER, [rates[n] for n in CONFIG_ORDER], color="0.4", edgecolor="black")
ax.set_ylabel("Hallucinated PMIDs per sample")
ax.set_title("Hallucinated PMID rate by config")
plt.xticks(rotation=15); plt.tight_layout(); plt.show()
rates

In [ ]:
# Per-sample B vs D scatter: x = # hallucinated PMIDs by B, y = by D
b = per_config["B_finetuned"].set_index("sample_idx")["hallucinated_pmids"].apply(len)
d = per_config["D_finetuned_rag"].set_index("sample_idx")["hallucinated_pmids"].apply(len)
joined = pd.concat([b.rename("B"), d.rename("D")], axis=1).fillna(0)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(joined["B"], joined["D"], alpha=0.6, color="0.3")
lim = max(joined.max().max(), 1)
ax.plot([0, lim], [0, lim], color="0.7", linestyle=":")
ax.set_xlabel("B hallucinated PMIDs"); ax.set_ylabel("D hallucinated PMIDs")
ax.set_title("Per-sample hallucinated PMIDs (B vs D)")
plt.tight_layout(); plt.show()

## 6. Format adherence

In [ ]:
vals = {n: children[n].data.metrics.get("format_adherence_among_answered", math.nan) for n in CONFIG_ORDER}
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(CONFIG_ORDER, [vals[n] for n in CONFIG_ORDER], color="0.4", edgecolor="black")
ax.set_ylabel("Fraction of answered"); ax.set_title("Format adherence (answered samples only)"); ax.set_ylim(0, 1.05)
plt.xticks(rotation=15); plt.tight_layout(); plt.show()
vals

## 7. RAGAS metrics with 95% CIs

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, metric in zip(axes.flat, ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]):
    means = [children[n].data.metrics.get(f"{metric}_mean", math.nan) for n in CONFIG_ORDER]
    lows = [children[n].data.metrics.get(f"{metric}_ci_low", math.nan) for n in CONFIG_ORDER]
    highs = [children[n].data.metrics.get(f"{metric}_ci_high", math.nan) for n in CONFIG_ORDER]
    n_used = [int(children[n].data.metrics.get(f"{metric}_n", 0)) for n in CONFIG_ORDER]
    lower = [m - lo if lo == lo else 0.0 for m, lo in zip(means, lows)]
    upper = [hi - m if hi == hi else 0.0 for m, hi in zip(means, highs)]
    ax.bar(CONFIG_ORDER, means, color="0.4", edgecolor="black", yerr=[lower, upper], capsize=4)
    for i, n in enumerate(n_used):
        ax.text(i, (means[i] or 0) + 0.02, f"n={n}", ha="center", fontsize=8)
    ax.set_title(metric.replace("_", " "))
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

## 8. Abstention analysis (C and D only)

In [ ]:
for name in ("C_base_rag", "D_finetuned_rag"):
    df = per_config[name]
    abst = df[df["abstained"]]
    print(f"{name}: {len(abst)}/{len(df)} abstained ({len(abst)/len(df):.1%})")
    for _, row in abst.head(2).iterrows():
        print(f"  idx={row['sample_idx']} | specialty={row['medical_specialty']} | trans[:120]={row['transcription'][:120]!r}")

## 9. Cross-config deltas (from rollup run)

In [ ]:
if rollup is None:
    print("No rollup run yet — run scripts/run_evaluation_rollup.py first.")
else:
    deltas = {k: v for k, v in rollup.data.metrics.items() if k.startswith("D_minus_")}
    pd.Series(deltas).to_frame("delta")

## 10. Failure case review

In [ ]:
for name, df in per_config.items():
    errs = df[df["error"].notna()]
    if not errs.empty:
        print(f"=== {name}: {len(errs)} errors ===")
        print(errs[["sample_idx", "medical_specialty", "error"]].to_string(index=False))

## 11. Cap. 6 narrative scratch

Free-form markdown space for drafting the thesis evaluation chapter against the numbers above.

- Headline claim: ___
- Key delta evidence: ___
- Caveats (ContextRecall ceiling, abstention as outcome): ___